In [73]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [74]:
import os
import pickle
import numpy as np
import pandas as pd
import scipy.stats as stats
from copy import deepcopy

from pickagm.corrmodels import (
    ClemettCorrelationModelAsc,
    ClemettCorrelationModelSInter,
    ClemettCorrelationModelSSlab,
    ClemettCorrelationModelVrancea,
    CORR_MODELS
)

from pickagm.selection import gcim_simulation
from pickagm.avgSA import indirect_AvgSA_GMPE

from openquake.hazardlib.gsim.base import registry
from openquake.hazardlib.imt import PGA, SA, RSD595, Sa_avg2, AvgSA
from openquake.hazardlib.gsim.mgmpe.generic_gmpe_avgsa import GenericGmpeAvgSA
from openquake.hazardlib.gsim.bahrampouri_2021_duration import (
    BahrampouriEtAldm2021Asc,
    BahrampouriEtAldm2021SInter,
    BahrampouriEtAldm2021SSlab
)

from phd_project.config.config import load_config 
from phd_project.scripts.oqhelpers import parse_nrml_logic_tree
from phd_project.scripts.WP1_ground_motion_set.gm_selection import (
    ESHM20SiteRupCtxBuilder
)

cfg = load_config()

In [75]:
# Load the disaggregation data
fp = cfg["proc_data"]["site_hazard_disaggregation"] / "AvgSA_03_disagg_data_60sites_5poes.pickle"
with open(fp, "rb") as f:
    disagg_data = pickle.load(f)

fp = cfg["proc_data"]["site_hazard_disaggregation"] / "AvgSA_03_disagg_stats_60sites_5poes.pickle"
with open(fp, "rb") as f:
    disagg_stats = pickle.load(f)

# load the site file
sites = pd.read_csv(cfg["hazard_models"]["eshm20_AvgSA_site_model_all"])

# load the flatfiles
flatfile_folder = cfg["proc_data"]["corr_model"] / "reverse" / "flatfiles"
flatfiles = {}
for f in [f for f in os.listdir(flatfile_folder) if f.endswith(".csv")]:
    tag = f.split("_")[0]
    flatfiles[tag] = pd.read_csv(flatfile_folder / f, delimiter=";", index_col=0)

flatfiles["volcanic"] = pd.read_csv(cfg["raw_data"]["gm_flatfiles"] / "volcanic_lanzanoluzi_flatfile.csv", 
                                    delimiter=";", index_col=0)


C:\Users\clemettn\AppData\Local\Temp\ipykernel_42180\1939585221.py:18: DtypeWarning: Columns (18,20,21,34,38,39,48,50) have mixed types. Specify dtype option on import or set low_memory=False.
  flatfiles[tag] = pd.read_csv(flatfile_folder / f, delimiter=";", index_col=0)


In [76]:
sites

,lat,lon,region,vs30,vs30measured,xvf,z1pt0,z2pt5
0,36.1,-5.31787,0,800,True,150.00,31.07,0.57
1,38.4,-0.51787,0,800,True,150.00,31.07,0.57
2,48.5,9.08213,0,800,True,150.00,31.07,0.57
3,47.2,18.48213,0,800,True,150.00,31.07,0.57
4,37.4,40.88213,0,800,True,150.00,31.07,0.57
5,50.8,6.08213,1,800,True,150.00,31.07,0.57
6,45.6,9.28213,1,800,True,150.00,31.07,0.57
7,36.8,14.48213,1,800,True,123.16,31.07,0.57
8,47.1,15.48213,1,800,True,150.00,31.07,0.57
9,43.8,18.38213,1,800,True,-150.00,31.07,0.57


In [77]:
disagg_stats

,site_id,lat,lon,seismicity,region,imt,poe,imtl,Craton [%],Non-Subduction Deep [%],Shallow Default [%],Subduction Inslab [%],Subduction Interface [%],Volcanic [%],Mag_mean,Dist_mean
0,30,36.1,-5.31787,high,0,AvgSA,0.020000,0.008389,0.0,0.0,98.02,1.98,0.0,0.0,5.98,49.75
1,30,36.1,-5.31787,high,0,AvgSA,0.002103,0.028372,0.0,0.0,99.73,0.27,0.0,0.0,6.22,11.79
2,30,36.1,-5.31787,high,0,AvgSA,0.000667,0.048784,0.0,0.0,99.96,0.04,0.0,0.0,6.30,10.15
3,30,36.1,-5.31787,high,0,AvgSA,0.000404,0.060800,0.0,0.0,99.98,0.02,0.0,0.0,6.32,10.07
4,30,36.1,-5.31787,high,0,AvgSA,0.000201,0.081782,0.0,0.0,99.99,0.01,0.0,0.0,6.35,10.03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,29,37.4,40.88213,lowmod,5,AvgSA,0.020000,0.009431,0.0,0.0,100.00,0.00,0.0,0.0,5.72,98.47
296,29,37.4,40.88213,lowmod,5,AvgSA,0.002103,0.027632,0.0,0.0,100.00,0.00,0.0,0.0,6.25,67.68
297,29,37.4,40.88213,lowmod,5,AvgSA,0.000667,0.042501,0.0,0.0,100.00,0.00,0.0,0.0,6.42,52.03
298,29,37.4,40.88213,lowmod,5,AvgSA,0.000404,0.050722,0.0,0.0,100.00,0.00,0.0,0.0,6.47,45.78


In [78]:
# a disaggregation distribution
site_id = 31
seismicity = "high"
region = 0
poe = 0.02
imt = "AvgSA"
sr = (seismicity, region)
disagg_dst = disagg_data[sr][site_id][imt][poe]
imtl = disagg_stats[(disagg_stats["site_id"] == site_id) &
                    (disagg_stats["seismicity"] == seismicity) &
                    (disagg_stats["region"] == region) &
                    (disagg_stats["imt"] == imt) &
                    (disagg_stats["poe"] == poe)]["imtl"].values[0]
disagg_type = "TRT_Mag_Dist_Eps"
assumed_rake = 0

In [79]:
def SA_PGA_gmm_from_logic_tree(LT: dict, trt: str):
    ignore_tags = ["avg_periods", "corr_func"]
    params = list(deepcopy(LT[trt]).values())[0]["params"]
    for tag in ignore_tags:
        params.pop(tag, None)
    gmpe_name = params.pop("gmpe_name")
    gmm = registry[gmpe_name](**params)
    return gmm

def AvgSA_gmm_from_logic_tree(LT: dict, trt: str):
        params = list(deepcopy(LT[trt]).values())[0]["params"]
        gmpe_name = params.pop("gmpe_name")
        periods = params.pop("avg_periods")
        corr_func = params.pop("corr_func")
        rho_total = CORR_MODELS[corr_func](incl_SA_Ts = periods).rho["total"]
        corr_mat = rho_total.to_numpy()
        base_gmm = registry[gmpe_name](**params)
        gmm = indirect_AvgSA_GMPE(base_gmm, corr_mat, avg_periods=periods)
        return gmm

In [80]:
# set some parameters for the selection
sa_periods = np.round(np.logspace(np.log10(0.025), np.log10(8), num=30), 3)
nonSA_imts = [AvgSA([0,3]).string, RSD595().string, PGA().string]  # strings match the correlation matrix
cond_imt = AvgSA([0,3])  # strings match the correlation matrix
selection_imts = [RSD595(), PGA()] + [SA(period) for period in sa_periods]  # these need to be imt instances to work with GMMs

# Create the GMM Map for AvgSA by reading the logic tree
fp = cfg["hazard_models"]["eshm20_AvgSA"] / "gmpe_logic_tree_AvgSA_0to3_median_branch.xml"
GMM_AvgSA_LT = parse_nrml_logic_tree(file_path=fp)

GMM_MAP = {
    "Craton": {
        "AvgSA": AvgSA_gmm_from_logic_tree(GMM_AvgSA_LT, "Craton"),         # these strings need to match the imt.name so that they be found when the imt is up to be calculated.
        "PGA": SA_PGA_gmm_from_logic_tree(GMM_AvgSA_LT, "Craton"),
        "SA": SA_PGA_gmm_from_logic_tree(GMM_AvgSA_LT, "Craton"),
        "RSD595": BahrampouriEtAldm2021Asc(),
        },
    "Non-Subduction Deep": {
        "AvgSA": AvgSA_gmm_from_logic_tree(GMM_AvgSA_LT, "Non-Subduction Deep"),
        "PGA": SA_PGA_gmm_from_logic_tree(GMM_AvgSA_LT, "Non-Subduction Deep"),
        "SA": SA_PGA_gmm_from_logic_tree(GMM_AvgSA_LT, "Non-Subduction Deep"),
        "RSD595": BahrampouriEtAldm2021SSlab()
        },
    "Shallow Default": {
        "AvgSA": AvgSA_gmm_from_logic_tree(GMM_AvgSA_LT, "Shallow Default"),
        "PGA": SA_PGA_gmm_from_logic_tree(GMM_AvgSA_LT, "Shallow Default"),
        "SA": SA_PGA_gmm_from_logic_tree(GMM_AvgSA_LT, "Shallow Default"),
        "RSD595": BahrampouriEtAldm2021Asc()
        },
    "Subduction Inslab": {
        "AvgSA": AvgSA_gmm_from_logic_tree(GMM_AvgSA_LT, "Subduction Inslab"),
        "PGA": SA_PGA_gmm_from_logic_tree(GMM_AvgSA_LT, "Subduction Inslab"),
        "SA": SA_PGA_gmm_from_logic_tree(GMM_AvgSA_LT, "Subduction Inslab"),
        "RSD595": BahrampouriEtAldm2021SSlab()
        },
    "Subduction Interface": {
        "AvgSA": AvgSA_gmm_from_logic_tree(GMM_AvgSA_LT, "Subduction Interface"),
        "PGA": SA_PGA_gmm_from_logic_tree(GMM_AvgSA_LT, "Subduction Interface"),
        "SA": SA_PGA_gmm_from_logic_tree(GMM_AvgSA_LT, "Subduction Interface"),
        "RSD595": BahrampouriEtAldm2021SInter()
        },
    "Volcanic": {
        "AvgSA": AvgSA_gmm_from_logic_tree(GMM_AvgSA_LT, "Volcanic"),
        "PGA": SA_PGA_gmm_from_logic_tree(GMM_AvgSA_LT, "Volcanic"),
        "SA": SA_PGA_gmm_from_logic_tree(GMM_AvgSA_LT, "Volcanic"),
        "RSD595": BahrampouriEtAldm2021Asc()
        }
}

# create the average depth map for each TRT #TODO:: if this needs to be more specific
average_depths = {
    "Craton": flatfiles["asc"]["ev_depth_km"].mean(),
    "Non-Subduction Deep": flatfiles["vran"]["ev_depth_km"].mean(),
    "Shallow Default": flatfiles["asc"]["ev_depth_km"].mean(),
    "Subduction Inslab": flatfiles["sinter"]["ev_depth_km"].mean(),
    "Subduction Interface": flatfiles["sinter"]["ev_depth_km"].mean(),
    "Volcanic": flatfiles["volcanic"]["ev_depth_km"].mean(),
}

# get the correlation model map
corr_model_map = {
    "Craton": ClemettCorrelationModelAsc(nonSA_imts, sa_periods).rho["total"],
    "Non-Subduction Deep": ClemettCorrelationModelVrancea(nonSA_imts, sa_periods).rho["total"],
    "Shallow Default": ClemettCorrelationModelAsc(nonSA_imts, sa_periods).rho["total"],
    "Subduction Inslab": ClemettCorrelationModelSSlab(nonSA_imts, sa_periods).rho["total"],
    "Subduction Interface": ClemettCorrelationModelSInter(nonSA_imts, sa_periods).rho["total"],
    "Volcanic": ClemettCorrelationModelAsc(nonSA_imts, sa_periods).rho["total"],
}

In [85]:
# create the site_rup context for the sampled M-R-TRT
site_params = sites.loc[site_id,:].to_dict()
ctx_builder = ESHM20SiteRupCtxBuilder(site_params, average_depths, assumed_rake)
sim_result = gcim_simulation(1, disagg_dst, disagg_type, imtl, GMM_MAP,
                             selection_imts, cond_imt, corr_model_map, ctx_builder,
                             1)


In [86]:
sim_result

[{'ctx': rec.array([(36.1, 28.08213, 0, 800,  True, 92.71, 31.07, 0.57, 'Subduction Inslab', 7.75, 170., 0.5, 30.96952, 0, 172.7979, 172.7979)],
            dtype=[('lat', '<f4'), ('lon', '<f4'), ('region', '<i4'), ('vs30', '<i4'), ('vs30measured', '?'), ('xvf', '<f4'), ('z1pt0', '<f4'), ('z2pt5', '<f4'), ('trt', '<U50'), ('mag', '<f4'), ('rjb', '<f4'), ('eps', '<f4'), ('hypo_depth', '<f4'), ('rake', '<i4'), ('rrup', '<f4'), ('rhypo', '<f4')]),
  'sim':                  mu    sig   mu_cond  sig_cond       sim
  RSD595     4.114386  0.458  4.258303  0.452748  4.414766
  PGA       -3.064959  0.740 -4.272267  0.459363 -3.974203
  SA(0.025) -2.967311  0.740 -4.151520  0.473044 -3.824953
  SA(0.031) -2.881981  0.740 -4.038472  0.488624 -3.768502
  SA(0.037) -2.814427  0.740 -3.943199  0.503369 -3.563398
  SA(0.045) -2.742285  0.740 -3.839488  0.519237 -3.441068
  SA(0.055) -2.639546  0.740 -3.709030  0.532425 -3.353623
  SA(0.068) -2.495567  0.740 -3.539026  0.544219 -3.121259
  SA(0.082) -

In [ ]:
## Simulation of IMTs for a given M-R-TRT
# # sample the disaggreation data to get a M-R-TRT values
# MRT_sample = disagg_dst.sample(1, weights=disagg_dst["P(m|X>x)"])
# trt = MRT_sample["TRT"].values[0]

# # create the site_rup context for the sampled M-R-TRT
# site_params = sites.loc[site_id,:].to_dict()
# ctx_builder = ESHM20SiteRupCtxBuilder(site_params, average_depths, assumed_rake)

# # initialize the AvgSA GMM with logic tree parameters
# gmm = GMM_MAP[trt][cond_imt.name]


# # calculate the expected ground motion for the sampled M-R-TRT
# mean = np.array([[0.0]])
# sig = np.zeros_like(mean)
# tau = np.zeros_like(mean)
# phi = np.zeros_like(mean)
# gmm.compute(ctx, [], mean, sig, tau, phi)

# # compare the expected to the disaggreagtion level and back-calculate epsilon
# eps = (np.log(imtl) - mean[0,0]) / sig[0,0]

# # calculate the conditional mean and sigma for the sampled M-R-TRT for all imts
# trt_gmms = GMM_MAP[trt]
# mu_cond = []
# sig_cond = []

# rho_mat = corr_model_map[trt]
# for imt in selection_imts:
#     gmm = trt_gmms[imt.name]
#     m = np.array([[0.0]])
#     s = np.zeros_like(m)
#     t = np.zeros_like(m)
#     p = np.zeros_like(mean)
#     gmm.compute(ctx, [imt], m, s, t, p)
    
#     rho = rho_mat.loc[imt.string, cond_imt.string]
#     mu_cond.append(m[0,0] + s[0,0] * rho * eps)
#     sig_cond.append(s[0,0] * np.sqrt(1 - rho ** 2))

# mu_cond = np.array(mu_cond).reshape(-1, 1)
# sig_cond = np.array(sig_cond).reshape(-1, 1)

# generate a random correlated vector
# rng = np.random.default_rng(seed=1)
# u = rng.standard_normal(len(selection_imts)).reshape(-1, 1)
# cond_rho_mat = conditional_correlation_matrix(rho_mat, cond_imt.string)
# L = np.linalg.cholesky(cond_rho_mat)
# v = L @ u

# # create a realisation of the the imt values for this M-R-TRT sample
# sim_imtls = mu_cond + sig_cond * v

In [ ]:
## Record Selection - Choose a GM that matches the simulated imts